# Suite CHG — Changes: what a transaction changed, observed at its commit

A journal, a viewer or a layout engine needs to know exactly what a transaction changed, which its expression, a query that writes, does not say ([CAP-270: a transaction's changes observed at its commit](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-270)). `Changes.observe(store, observer)` gives `observer` the effect of each transaction that commits having changed something, and returns a function that ends it. The effect is written in the journal language: a transaction expression of the store's own schemas, one upsert, whose symbols are bound to the objects held before by their references and to the objects created by their schemas, and whose spec removes and inserts relation entries and sets and clears properties, net of what the transaction undid within itself. Applied to the store as it was, it makes the same changes; applied again, none.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Changes, Plain, Proxies
from support import raises


def crm(txn):
    """crm.Person, owned by the singleton book by key, with a name and a home, who knows persons since a year, and
    pins persons through no adjacency by name; crm.Note, counted, about a person."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Home": S.OfObject, "Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation,
                  "Book": S.OfObject, "Note": S.OfObject, "About": S.OfRelation, "Pinned": S.OfRelation})
        .spec([
            lambda _: _.Home.name("crm.Home").properties({"city": S.OfStr, "zip": S.OfStr}),
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")])
            .properties({"since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "home": _.Home}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
                lambda t: t.relation([_.Pinned]).me("pinner"),  # no adjacency to crm.Pinned has a name
                lambda t: t.relation([_.Pinned]).me("pinned"),
            ]),
            lambda _: _.Pinned.name("crm.Pinned").links([lambda t: t.name("pinner"), lambda t: t.name("pinned")]),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")])
            .properties({"key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book = store.registered("crm.Person"), store.registered("crm.Book")
book = store.singleton("crm.TheBook")


def short(path):
    """A path, short: its singleton, then each key it passes, `crm.TheBook/ada`; a schema's, its name."""
    parts, at = [path["singleton"]], path.get("next", {})
    while "entry" in at:
        parts += [str(v) for v in at["entry"].get("properties", {}).values()]
        at = at["entry"].get("next", {})
    return ".".join(parts[1:]) if parts[0] == "." else "/".join(parts)


def shown(effect, source=None):
    """An effect of `source`, the crm store unless given, short: each symbol, as what it is bound to, then each mutation
    of its spec, in order."""
    (pattern,) = json.loads(JSON.dumps(Plain.ToPlain(source or store).Transaction(effect)))["mutations"][0]["upsert"]["exists"]
    lines = []
    for name, bound in pattern["symbols"].items():
        lines.append(f"{name} = {short(bound['reference']['path'])}" if "reference" in bound
                     else f"{name} = new {short(bound['schema'][0]['$ref'])}")
    for spec in pattern["spec"]:
        for name, mutations in spec.items():
            for m in mutations:
                if "as_property" in m:
                    (verb,) = m["as_property"]["mutations"]
                    lines.append(f"{name}.{m['as_property']['name']} = " + (json.dumps(verb["set"]["value"]) if "set" in verb
                                                                           else "cleared"))
                    continue
                for v in m["as_adjacency"]["mutations"]:
                    ((kind, body),) = v.items()
                    for entry in body["entries"]:
                        parts = []
                        for x in entry["mutations"]:
                            if "as_link" in x:
                                parts.append(f"{x['as_link']['name']}={x['as_link']['mutations'][0]['set']['ref']['symbol']}")
                            else:
                                parts.append(f"{x['as_property']['name']}={json.dumps(x['as_property']['mutations'][0]['set']['value'])}")
                        lines.append(f"{'+' if kind == 'insert' else '-'} {name}.{m['as_adjacency']['name']}({', '.join(parts)})")
    return lines


def people(*names):
    """A transaction adding persons to the book, each keyed by its name."""
    return lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"book": Book, **{n: Person for n in names}})
        .spec([lambda _: _.book.contacts({n: (lambda e, n=n: e.contact(getattr(_, n))) for n in names}),
               *[(lambda _, n=n: getattr(_, n).name(n)) for n in names]])]))])


log = []
end = Changes.observe(store, log.append)

<a id="CHG-01"></a>
## CHG-01 · Each transaction that commits having changed something is given to the observer as its effect, after its commit; one that changes nothing is not

In [ ]:
store.transaction(people("ada", "bob"))
assert shown(log[-1]) == ["o0 = crm.TheBook", "o1 = new crm.Person", "o2 = new crm.Person",
                          '+ o0.contacts(contact=o1, key="ada")', '+ o0.contacts(contact=o2, key="bob")',
                          'o1.name = "ada"', 'o2.name = "bob"']
assert log[-1].as_schema() is store.registered("Mutations.OfTransaction")  # schema data, as any value is
count = len(log)
store.transaction(people("ada"))  # changes nothing: the spec holds
assert len(log) == count

<a id="CHG-02"></a>
## CHG-02 · A transaction that raises, its writes dropped, is observed by none

In [ ]:
ada = book.contacts({"key": "ada"})
count = len(log)
with raises(RuntimeError, match="refused"):
    with store.transaction():
        store.crm.Person(ada, lambda p: p.name("Zed"))
        raise RuntimeError("refused")
assert len(log) == count and ada.name.get() == "ada"

<a id="CHG-03"></a>
## CHG-03 · What a transaction undid within itself is no change: an entry inserted and removed, a value set and set back

In [ ]:
bob = book.contacts({"key": "bob"})
count = len(log)
with store.transaction():
    store.crm.Person(ada, lambda p: p.knows(lambda a: a.insert(lambda e: e.known(bob).since(2001))))
    store.crm.Person(ada, lambda p: p.knows(lambda a: a.remove(lambda e: e.known(bob))))
    store.crm.Person(ada, lambda p: p.name("Ada"))
    store.crm.Person(ada, lambda p: p.name("ada"))
assert len(log) == count

<a id="CHG-04"></a>
## CHG-04 · Removing an owning entry deletes what it owned: the effect removes that entry by its owner's key, naming nothing deleted, and what goes with it follows

In [ ]:
with store.transaction():
    store.crm.Person(ada, lambda p: p.knows(lambda a: a.insert(lambda e: e.known(bob).since(2001))))
assert shown(log[-1]) == ["o0 = crm.TheBook/bob", "o1 = crm.TheBook/ada", "+ o1.knows(known=o0, since=2001)"]
with store.transaction():
    store.crm.Book(book, lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("bob"))))
assert shown(log[-1]) == ["o0 = crm.TheBook", '- o0.contacts(key="bob")']

<a id="CHG-05"></a>
## CHG-05 · A property changed is set to its value, written whole by its schema, a value object's part included, or cleared

In [ ]:
with store.transaction():
    store.crm.Person(ada, lambda p: p.home(lambda h: h.city("Paris").zip("75001")))
with store.transaction():
    store.crm.Person(ada, lambda p: p.home(lambda h: h.city("Lyon")))
assert shown(log[-1]) == ["o0 = crm.TheBook/ada", 'o0.home = {"city": "Lyon", "zip": "75001"}']
with store.transaction():
    store.crm.Person(ada, lambda p: p.name(None))
assert shown(log[-1]) == ["o0 = crm.TheBook/ada", "o0.name = cleared"]

<a id="CHG-06"></a>
## CHG-06 · A counted object, immutable, is written by value: bound to its schema, its properties and entries written, found by them

In [ ]:
with store.transaction():
    store.crm.Note(lambda n: n.text("hello").about(lambda a: a.insert(lambda e: e.person(ada))))
assert shown(log[-1]) == ["o0 = crm.TheBook/ada", "o1 = new crm.Note", "+ o1.about(person=o0)", 'o1.text = "hello"']

<a id="CHG-07"></a>
## CHG-07 · Each effect, written and read as plain data and applied in order to a store as the first was, makes it the same, applied twice the same again; once the observing ends, none is given

In [ ]:
replica = B.Store()
replica.transaction(crm)
for effect in log:
    plain = json.loads(JSON.dumps(Plain.ToPlain(store).Transaction(effect)))
    for _ in range(2):
        replica.transaction(Plain.FromPlain(replica).Transaction(plain))
data = lambda s: JSON.dumps(Plain.ToPlain(s).Snapshot()[1])  # its data, its schemas aside
assert data(replica) == data(store)
end()
end()  # ending it again changes nothing
count = len(log)
store.transaction(people("eve"))
assert len(log) == count

<a id="CHG-08"></a>
## CHG-08 · An entry that no object it links names an adjacency to cannot be written in the effect: while observed, its transaction is refused, naming its relation

In [ ]:
pinned = store.registered("crm.Pinned")
stop = Changes.observe(store, lambda effect: None)
with raises(ValueError, match="no object an entry of 'crm.Pinned' links has an adjacency to it by name"):
    with store.transaction():  # only the framework's own writes reach such a relation
        Proxies._add_entry(pinned, Proxies._Entry(pinned, {"pinner": ada, "pinned": ada}, {}))
stop()
assert not Proxies._linking(pinned, "pinner", ada)

<a id="CHG-09"></a>
## CHG-09 · Objects of many schemas owned through one relation: deleting one is written by its owner's key alone, with what it owned and every entry linking it following, applied twice the same; an owner with no adjacency by name to the relation cannot write it

In [ ]:
def org(txn):
    """org.Team and org.Member, each owned through org.Owns, by name, as is org.Badge, a member's; members are buddies;
    org.Key is owned by the singleton vault through no adjacency by name."""
    S = txn.Schemas

    def owned(t, _):
        return t.name("owner").relation([_.Owns]).me("owned").owner(True)

    return txn.uid("org/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Owns": S.OfRelation, "Root": S.OfObject, "Team": S.OfObject, "Member": S.OfObject,
                  "Badge": S.OfObject, "Buddy": S.OfRelation, "Holds": S.OfRelation, "Vault": S.OfObject,
                  "Key": S.OfObject})
        .spec([
            lambda _: _.Owns.name("org.Owns").links([lambda t: t.name("owner"), lambda t: t.name("owned")])
            .properties({"name": S.OfStr}).uniques([["owner", "name"], ["owned"]]),
            lambda _: _.Buddy.name("org.Buddy").links([lambda t: t.name("a"), lambda t: t.name("b")]),
            lambda _: _.Holds.name("org.Holds").links([lambda t: t.name("vault"), lambda t: t.name("key")])
            .properties({"name": S.OfStr}).uniques([["vault", "name"], ["key"]]),
            lambda _: _.Root.name("org.Root").ref(True).singleton("org.TheRoot").adjacencies({
                "owns": lambda t: t.relation([_.Owns]).me("owner")}),
            lambda _: _.Team.name("org.Team").ref(True).adjacencies([
                lambda t: owned(t, _), lambda t: t.name("owns").relation([_.Owns]).me("owner")]),
            lambda _: _.Member.name("org.Member").ref(True).adjacencies([
                lambda t: owned(t, _), lambda t: t.name("owns").relation([_.Owns]).me("owner"),
                lambda t: t.name("buddies").relation([_.Buddy]).me("a"),
                lambda t: t.relation([_.Buddy]).me("b")]),
            lambda _: _.Badge.name("org.Badge").ref(True).properties({"code": S.OfStr}).adjacencies([
                lambda t: owned(t, _)]),
            lambda _: _.Vault.name("org.Vault").ref(True).singleton("org.TheVault").adjacencies([
                lambda t: t.relation([_.Holds]).me("vault")]),  # no adjacency by name
            lambda _: _.Key.name("org.Key").ref(True).adjacencies([
                lambda t: t.name("vault").relation([_.Holds]).me("key").owner(True)]),
        ])]))])


first, second = B.Store(), B.Store()
for s in (first, second):
    s.transaction(org)
effects = []
Changes.observe(first, effects.append)
root = first.singleton("org.TheRoot")
with first.transaction():
    team = first.org.Team(lambda t: t.owner(lambda a: a.insert(lambda e: e.name("core").owner(root))))
    ann = first.org.Member(lambda m: m.owner(lambda a: a.insert(lambda e: e.name("ann").owner(team))))
    ben = first.org.Member(lambda m: m.owner(lambda a: a.insert(lambda e: e.name("ben").owner(team)))
                           .buddies(lambda a: a.insert(lambda e: e.b(ann))))
    first.org.Badge(lambda b: b.owner(lambda a: a.insert(lambda e: e.name("pass").owner(ann))).code("A1"))
with first.transaction():
    first.org.Member(ben, lambda m: m.buddies(lambda a: a.remove(lambda e: e.b(ann))))
assert shown(effects[-1], first) == ["o0 = org.TheRoot/core/ann", "o1 = org.TheRoot/core/ben", "- o1.buddies(b=o0)"]
with first.transaction():
    first.org.Member(ben, lambda m: m.buddies(lambda a: a.insert(lambda e: e.b(ann))))
with first.transaction():
    first.org.Team(team, lambda t: t.owns(lambda a: a.remove(lambda e: e.name("ann"))))
assert shown(effects[-1], first) == ["o0 = org.TheRoot/core", '- o0.owns(name="ann")']
for effect in effects:
    plain = json.loads(JSON.dumps(Plain.ToPlain(first).Transaction(effect)))
    for _ in range(2):
        second.transaction(Plain.FromPlain(second).Transaction(plain))
assert data(second) == data(first) and [e.name.get() for e in team.owns] == ["ben"] and not list(ben.buddies)
vault = first.singleton("org.TheVault")
with first.transaction():
    key = first.org.Key(lambda k: k.vault(lambda a: a.insert(lambda e: e.name("k1").vault(vault))))
with raises(ValueError, match="no owner of an entry of 'org.Holds' has an adjacency to it by name"):
    with first.transaction():
        first.org.Key(key, lambda k: k.vault(lambda a: a.clear()))